# R&D 3-B: On-Device Deployment — OpenVINO INT8 Quantization

**목적**: TFLite 변환 이슈를 겪은 후, 대안으로 Intel OpenVINO 프레임워크를 활용한 모델 최적화를 다각도로 검토.

**시도한 내용**:
- `optimum-cli`로 `elmenwol/whisper-small_aihub_child` 모델을 OpenVINO IR 포맷(INT8)으로 변환
- `openvino_genai.WhisperPipeline`으로 CPU 추론 테스트
- ONNX, TFLite, OpenVINO 세 프레임워크 간 비교 검토

**결론**: OpenVINO 변환 및 CPU 추론 자체는 성공. 그러나 모바일 배포 환경과의 호환성, 모델 정확도 유지 및 추론 속도 문제를 종합적으로 판단하여 클라우드 서버사이드 추론으로 최종 결정.


In [1]:
!pip install  -U "openvino>=2024.3.0" "openvino-genai"
!pip install "torch>=2.1" "nncf>=2.7" "transformers>=4.40.0" "onnx<1.16.2" "optimum>=1.16.1" "accelerate" "datasets>=2.14.6" "git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 48.3 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
  Cloning https://github.com/huggingface/optimum-intel.git to /tmp/pip-req-build-7mxlaot9
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/optimum-intel.git /tmp/pip-req-build-7mxlaot9
  Resolved https://github.com/huggingface/optimum-intel.git to commit c94b3f5efaedb8c83cb15f8e0cede6f523631bde
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 7.4 MB/s eta 0:00:00
  Preparing 

In [2]:
!pip install optimum[openvino,nncf]

In [3]:
!pip install notebook_utils openvino_genai

In [4]:
!optimum-cli export openvino --model elmenwol/whisper-small_aihub_child --task automatic-speech-recognition-with-past --weight-format int8 asr_openvino_int8

2024-12-04 14:14:54.394369: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-04 14:14:54.439669: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-04 14:14:54.452826: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-04 14:14:54.481847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-04 14:14:57.233803: W tensorflow/compiler/tf2

In [5]:
import openvino_genai as ov_genai

ov_pipe = ov_genai.WhisperPipeline(str('/content/asr_openvino_int8'), device='CPU')

In [9]:
import time
import librosa

In [14]:
start = time.time()
raw_speech, samplerate = librosa.load(str('/content/서울의 중심에는 한강 하류가 동에서 .wav'), sr=16000)
transcription = ov_pipe.generate(raw_speech)

end = time.time()
print(f"Transcription time: {end - start} seconds")
print(transcription)

Transcription time: 13.969604015350342 seconds
서울의 중심에는 한강하류가 동에서 서쪽으로 흐르고 있다.


In [11]:
import os
from transformers import WhisperProcessor, logging as transformers_log
from optimum.intel.openvino import OVModelForSpeechSeq2Seq
import torchaudio
import torch
import numpy as np
import time

# from src import log
# from src.utils import utils

import asyncio

class Log:
    @staticmethod
    def info(message):
        print(f"[INFO] {message}")

    @staticmethod
    def error(message):
        print(f"[ERROR] {message}")

log = Log()

class OpenVinoService:
    _initialized = False

    def __init__(self, language='ko'):
        if not OpenVinoService._initialized:
            os.environ["TRANSFORMERS_VERBOSITY"] = "error"
            transformers_log.set_verbosity_error()
            self.model_name = 'elmenwol/whisper-small_aihub_child'
            self.ov_model_name = '/content/asr_openvino_int8'
            self.language = language
            self.task = 'transcribe'
            self.device = "CPU"
            self.sr = 16000

            try:
                # Initialize model and related components
                log.info("Starting OpenVino service...")
                self.model = self.get_openvino_model()
                self.compile_openvino_model()
                self.processor = self.create_processor()

                OpenVinoService._initialized = True
                log.info("OpenVino service started with success!")
            except Exception as e:
                log.error(f"Error during OpenVino service init: {str(e)}")
                raise

    def get_openvino_model(self):
        """
        """
        ov_config = {"CACHE_DIR": ""}
        self.model = OVModelForSpeechSeq2Seq.from_pretrained(self.ov_model_name, ov_config=ov_config, compile=False)
        log.info("OpenVino model loaded from " + str(self.ov_model_name))

        return self.model


    def compile_openvino_model(self):
        """
        """
        try:

            if torch.cuda.is_available():
                log.info("Model loaded on GPU")
                self.device = "GPU"
            else:
                log.info("Model loaded on CPU")
                self.device = "CPU"

            self.model.to(self.device)
            self.model.compile()

            log.info("OpenVino model compiled successfully")

        except Exception as e:
            log.error(f"Error during OpenVino model compilation: {str(e)}")
            raise

        return self.model


    def create_processor(self):
        """
        """
        try:
            processor = WhisperProcessor.from_pretrained(
                self.model_name,
                language=self.language,
                task=self.task
            )
            log.info("WhisperProcessor created")
            return processor
        except Exception as e:
            log.error(f"Error during WhisperProcessor creation: {str(e)}")
            raise


    def preprocess_audio(self, waveform):
        """
        """
        # compute log-Mel input features from input audio array
        audio_features = self.processor.feature_extractor(waveform, sampling_rate=self.sr).input_features[0]
        audio_features = torch.tensor(np.array([audio_features]))

        return audio_features


    def openvino_pipeline(self,audio_path):

        # print("1 - starting audio load:", time.time())
        waveform, sample_rate = torchaudio.load(audio_path)
        waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=self.sr)(waveform)[0]
        # print("2 - starting preprocessing:", time.time())
        audio_features = self.preprocess_audio(waveform)

        # print("3 - starting forward pass:", time.time())
        predicted_ids = self.model.generate(audio_features, max_new_tokens=224)

        # print("4 - starting decoding:", time.time())
        transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)

        return transcription[0]


    async def transcribe(self, audio_path: str) -> str:
        """
        """
        try:
            loop = asyncio.get_event_loop()
            log.info(f"Transcribing the following file audio: {audio_path}")

            print("0 - starting the loop:",time.time())
            text = await loop.run_in_executor(
                None,
                lambda: self.openvino_pipeline(audio_path)
                )

            print("5 - all done:", time.time())
            log.info("Transcription completed!")
            return text

        except Exception as e:
            log.error(f"Error during transcription: {str(e)}")
            raise

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


In [12]:
whisper = OpenVinoService()

[INFO] Starting OpenVino service...
[INFO] OpenVino model loaded from /content/asr_openvino_int8
[INFO] Model loaded on CPU
[INFO] OpenVino model compiled successfully


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[INFO] WhisperProcessor created
[INFO] OpenVino service started with success!


In [ ]:
start = time.time()
transcription = await whisper.transcribe('/content/서울의 중심에는 한강 하류가 동에서 .wav')
end = time.time()
log.info("Transcription completed")
print(f"Transcription time: {end - start} seconds")
print(transcription)

[INFO] Transcribing the following file audio: /content/서울의 중심에는 한강 하류가 동에서 .wav
0 - starting the loop: 1733322057.637153
